# Food-101 (20 classes): CNN trained from scratch

**Owner:** Trung Le

A 4-block convolutional network trained from random weights. Data split, training loop and metrics come from `common.py`, shared with the ResNet-50 notebook.

**On Google Colab:** Runtime → Change runtime type → GPU, then run this cell. Skip it when running locally.

In [ ]:
# !git clone https://github.com/Tinle0301/food101-cecs456.git
# %cd food101-cecs456
# !pip install -q -r requirements.txt

In [ ]:
import torch
import torch.nn as nn
from common import *

set_seed()
print('Device:', get_device())

## 1. Data
Downloads Food-101 the first time (~5 GB) and creates `split.json`: 600 train / 150 validation / 250 test images per class.

In [ ]:
# Set QUICK_TEST = True to check everything runs on a small subset first
QUICK_TEST = False
LIMIT = 300 if QUICK_TEST else 0

In [ ]:
IMG_SIZE, BATCH_SIZE = 128, 64
loaders = get_loaders(IMG_SIZE, BATCH_SIZE, limit=LIMIT)
print({k: len(v.dataset) for k, v in loaders.items()})

## 2. Model
Four blocks of (3×3 conv → BatchNorm → ReLU) ×2 → max pooling, with 32 → 64 → 128 → 256 channels, then global average pooling, dropout and a linear layer for the 20 classes.

In [ ]:
def conv_block(c_in, c_out):
    return nn.Sequential(
        nn.Conv2d(c_in, c_out, 3, padding=1, bias=False), nn.BatchNorm2d(c_out), nn.ReLU(inplace=True),
        nn.Conv2d(c_out, c_out, 3, padding=1, bias=False), nn.BatchNorm2d(c_out), nn.ReLU(inplace=True),
        nn.MaxPool2d(2))


class ScratchCNN(nn.Module):
    def __init__(self, num_classes=20):
        super().__init__()
        self.features = nn.Sequential(conv_block(3, 32), conv_block(32, 64),
                                      conv_block(64, 128), conv_block(128, 256))
        self.classifier = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(),
                                        nn.Dropout(0.4), nn.Linear(256, num_classes))

    def forward(self, x):
        return self.classifier(self.features(x))


model = ScratchCNN(len(CLASSES))
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

## 3. Training
AdamW (lr 1e-3, weight decay 1e-4) with a cosine learning-rate schedule. The weights from the epoch with the best validation macro-F1 are kept.

In [ ]:
EPOCHS = 2 if QUICK_TEST else 30
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
hist = train_model(model, loaders, optimizer, EPOCHS, 'scratch', scheduler)

In [ ]:
plot_curves(hist, 'Scratch CNN');

## 4. Test set results

In [ ]:
res = evaluate(model, loaders, 'scratch')

In [ ]:
plot_confusion(res, 'Scratch CNN');

In [ ]:
plot_top_losses(res, loaders, 'Scratch CNN');

## 5. Save results
Downloads a zip with the learning curves and test metrics (Colab deletes files when the session ends). Unzip it into your local repo folder, then commit and push.

In [ ]:
download_results('scratch')